In [1]:
import numpy as np
from sgp4.api import Satrec
import matplotlib.pyplot as plt
import plotly.graph_objects as go


# ===============================
# INPUT TLE
# ===============================

line1 = "1 25544U 98067A   24123.50000000  .00016717  00000+0  10270-3 0  9991"
line2 = "2 25544  51.6416 130.5360 0006703  85.3290  37.2566 15.50012345678901"



In [2]:
# ===============================
# CREATE SATELLITE OBJECT
# ===============================

satellite = Satrec.twoline2rv(line1, line2)

# ===============================
# GET STATE VECTOR AT TLE EPOCH
# ===============================

jd = satellite.jdsatepoch
fr = satellite.jdsatepochF

error_code, r, v = satellite.sgp4(jd, fr)

# ===============================
# OUTPUT
# ===============================

if error_code == 0:
    r = np.array(r)
    v = np.array(v)

    print("Position Vector (km):")
    print(r)

    print("\nVelocity Vector (km/s):")
    print(v)
else:
    print("SGP4 propagation error:", error_code)

Position Vector (km):
[ -313.51748119 -5089.21846027  4477.27068805]

Velocity Vector (km/s):
[ 6.14653269 -3.24251386 -3.2420658 ]


In [3]:

# ==========================================
# INPUT STATE VECTOR
# ==========================================

#r = np.array([6521.3, -1324.8, 1022.4])   # km
Re = 6378.137                              # km

# ==========================================
# CREATE EARTH SPHERE
# ==========================================

u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 60)

x = Re * np.outer(np.cos(u), np.sin(v))
y = Re * np.outer(np.sin(u), np.sin(v))
z = Re * np.outer(np.ones(len(u)), np.cos(v))

earth = go.Surface(
    x=x,
    y=y,
    z=z,
    opacity=0.5,
    showscale=False,
    name='Earth'
)

# ==========================================
# TEME AXES
# ==========================================

axis_len = 8000

x_axis = go.Scatter3d(
    x=[0, axis_len],
    y=[0, 0],
    z=[0, 0],
    mode='lines+text',
    text=['', 'TEME X'],
    line=dict(width=6),
    name='X-axis'
)

y_axis = go.Scatter3d(
    x=[0, 0],
    y=[0, axis_len],
    z=[0, 0],
    mode='lines+text',
    text=['', 'TEME Y'],
    line=dict(width=6),
    name='Y-axis'
)

z_axis = go.Scatter3d(
    x=[0, 0],
    y=[0, 0],
    z=[0, axis_len],
    mode='lines+text',
    text=['', 'TEME Z'],
    line=dict(width=6),
    name='Z-axis'
)

# ==========================================
# SATELLITE VECTOR
# ==========================================

sat_vector = go.Scatter3d(
    x=[0, r[0]],
    y=[0, r[1]],
    z=[0, r[2]],
    mode='lines+markers+text',
    text=['Earth Center', 'Satellite'],
    line=dict(width=8),
    marker=dict(size=[5, 8]),
    name='State Vector'
)

# ==========================================
# FIGURE
# ==========================================

fig = go.Figure(data=[
    earth,
    x_axis,
    y_axis,
    z_axis,
    sat_vector
])

fig.update_layout(
    title='Interactive TEME Coordinate System',
    scene=dict(
        xaxis_title='X (km)',
        yaxis_title='Y (km)',
        zaxis_title='Z (km)',
        aspectmode='data'
    ),
    width=900,
    height=900
)

fig.show()